# Eixo 2 - Prática de Ciência de Dados: Pré-processamento (SINAN Esquistossomose)

### Membros
- Luiz Gabriel Correia dos Santos (15639682)
- Ana Paula de Abreu Batista (12688424)
- Italo Carlos Martins Bresciani (15461782)
- Arthur Filliettaz
- Arthur Araujo

## Objetivo deste notebook
Construir um pipeline completo de pré-processamento para o arquivo `sinan_esqu.csv`, com base:

1. No entendimento dos atributos levantado no notebook de EDA (`esqu_EDA.ipynb`).
2. No descarte imediato definido de colunas irrelevantes.

## Etapas que serão cobertas
1. Carregamento e inspeção inicial dos dados.
2. Descarte inicial de colunas irrelevantes.
3. Padronização de dados faltantes.
4. Conversão de datas em formato consistente.
5. Engenharia de atributos (features temporais).
6. Tratamento de inconsistências de domínio.
7. Tratamento de códigos categóricos de ignorado.
8. Imputação (mediana/moda).
9. Codificação para modelagem (One-Hot + Frequency Encoding).
10. Padronização e PCA (opcional).
11. Exportação das bases finais.

## 0. Imports e Configurações

In [1]:
# Bibliotecas base para manipulação e análise tabular
import numpy as np
import pandas as pd
from pathlib import Path

# Bibliotecas de pré-processamento para modelagem
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Configurações para facilitar inspeção no notebook
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)

print('Bibliotecas importadas com sucesso.')

Bibliotecas importadas com sucesso.


## 1. Carregamento e Inspeção Inicial

In [2]:
# Carrega os dados de esquistossomose diretamente do SINAN (via PySUS)
# !pip install pysus
from pysus import SINAN
import pandas as pd
import numpy as np

sinan = SINAN().load()
files = sinan.get_files(dis_code=["ESQU"])
parquets = sinan.download(files)
print(f'Arquivos baixados: {len(parquets)}')

dfs = [p.to_dataframe() for p in parquets]
df_raw = pd.concat(dfs, ignore_index=True)

# Mantém somente casos positivos para esquistossomose (AN_QUALI == 1)
df_raw['AN_QUANT'] = (
    df_raw['AN_QUANT']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA})
)

print('Distribuição AN_QUANT antes do filtro:')
display(df_raw['AN_QUANT'].value_counts(dropna=False))

antes = len(df_raw)
df_raw = df_raw[df_raw['AN_QUANT'] == '1'].copy()
apos_anquali = len(df_raw)

# Remove linhas sem alvo válido para modelagem supervisionada de EVOLUCAO
df_raw['EVOLUCAO'] = (
    df_raw['EVOLUCAO']
    .astype('string')
    .str.strip()
    .replace({'': pd.NA, '9': pd.NA, '99': pd.NA, 'Ignorado': pd.NA, 'IGNORADO': pd.NA})
)
df_raw = df_raw[df_raw['EVOLUCAO'].notna()].copy()
apos_evolucao = len(df_raw)

print(f'Registros antes dos filtros: {antes:,}')
print(f'Registros após AN_QUANT == 1: {apos_anquali:,}')
print(f'Registros após remover EVOLUCAO inválida: {apos_evolucao:,}')

print('Distribuição EVOLUCAO após filtros:')
display(df_raw['EVOLUCAO'].value_counts(dropna=False))

df_raw.head()

78882it [00:00, 2489672.65it/s]       


Arquivos baixados: 19
Distribuição AN_QUANT antes do filtro:


AN_QUANT
1       84062
0       79842
<NA>     2961
Name: count, dtype: Int64

Registros antes dos filtros: 166,865
Registros após AN_QUANT == 1: 84,062
Registros após remover EVOLUCAO inválida: 64,456
Distribuição EVOLUCAO após filtros:


EVOLUCAO
1    63277
2      811
3      257
4      111
Name: count, dtype: Int64

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_SIN_PRI,SEM_PRI,ANO_NASC,NU_IDADE_N,CS_SEXO,CS_GESTANT,CS_RACA,CS_ESCOL_N,SG_UF,ID_MN_RESI,ID_RG_RESI,ID_PAIS,DT_INVEST,ID_OCUPA_N,DT_COPRO,AN_QUANT,OUTROS,TRATAM,TRATANAO,STCURA1,STCURA2,STCURA3,FORMA,TPAUTOCTO,COUFINF,COPAISINF,COMUNINF,NOPROPIN,NOCOLINF,DOENCA_TRA,EVOLUCAO,DT_ENCERRA,DT_DIGITA,DT_TRANSUS,DT_TRANSDM,DT_TRANSSM,DT_TRANSRS,DT_TRANSSE,DT_OBITO,DS_FORMA,AN_QUALI,DTTRAT,DT_RESU3,OUTRO_EX
0,2,B659,2007-10-30,200744,2007,26,260765,1507,2429020,2007-07-11,200728,1983,4024,F,5,1,02,26,260765,1507,1,20071112,,20071107,1,3,1,,,,,1,1,26,1,260765,,,9,1,20071226,20080812,,,,20080820,20081010,,,1,20071122,,
6,2,B659,2007-10-08,200741,2007,26,260500,1499,2631504,2007-08-08,200732,1983,4023,M,6,4,03,26,260500,1499,1,20071008,,20070924,1,,3,9,,,,1,1,26,1,260500,,,2,1,20071009,20071227,,,,,20080118,,,1,,,
7,2,B659,2007-10-18,200742,2007,26,260420,1498,2712741,2007-08-15,200733,1980,4026,F,5,1,01,26,260420,1498,1,20071018,999992,20071008,1,3,1,,2,2,2,1,1,26,1,260420,,,9,1,20071018,20071019,,,,20071026,20080121,,,1,20071018,,
8,2,B659,2007-10-09,200741,2007,26,260420,1498,2712741,2007-08-10,200732,1997,4009,M,6,4,01,26,260420,1498,1,20071009,999991,20070829,1,3,1,,2,2,2,1,1,26,1,260420,,,9,1,20071009,20071019,,,,20071026,20080121,,,1,20071009,,
9,2,B659,2007-10-02,200740,2007,26,260420,1498,2712741,2007-09-27,200739,1973,4034,F,5,4,01,26,260420,1498,1,20071002,,20070928,1,3,1,,2,2,2,1,1,26,1,260420,,,9,1,20071002,20071004,,,,20071026,20080121,,,1,20071002,,


In [3]:
print("Value counts para 'AN_QUALI' em df_raw (após filtros):")
display(df_raw['AN_QUALI'].value_counts(dropna=False))

print("Value counts para 'EVOLUCAO' em df_raw (após filtros):")
display(df_raw['EVOLUCAO'].value_counts(dropna=False))

Value counts para 'AN_QUALI' em df_raw (após filtros):


AN_QUALI
1    49611
3    10020
      4264
2      561
Name: count, dtype: Int64

Value counts para 'EVOLUCAO' em df_raw (após filtros):


EVOLUCAO
1    63277
2      811
3      257
4      111
Name: count, dtype: Int64

### Removemos EVOLUCAO = 4  pois são casos de óbitos por outras causas, o que não é o foco do nosso estudo.

In [4]:
print('Distribuição EVOLUCAO antes do filtro (removendo 4):')
display(df_raw['EVOLUCAO'].value_counts(dropna=False))

antes_filtro_4 = len(df_raw)
df_raw = df_raw[df_raw['EVOLUCAO'] != '4'].copy()

print(f'Registros antes de remover EVOLUCAO == 4: {antes_filtro_4:,}')
print(f'Registros após remover EVOLUCAO == 4: {len(df_raw):,}')

print('Distribuição EVOLUCAO após o filtro (removendo 4):')
display(df_raw['EVOLUCAO'].value_counts(dropna=False))

Distribuição EVOLUCAO antes do filtro (removendo 4):


EVOLUCAO
1    63277
2      811
3      257
4      111
Name: count, dtype: Int64

Registros antes de remover EVOLUCAO == 4: 64,456
Registros após remover EVOLUCAO == 4: 64,345
Distribuição EVOLUCAO após o filtro (removendo 4):


EVOLUCAO
1    63277
2      811
3      257
Name: count, dtype: Int64

### 1.1 Diagnóstico inicial de faltantes

Nesta etapa calculamos a taxa de ausência por atributo, usando:

$$
\text{Taxa de faltantes}(j) = \frac{\#\{x_{ij}\,\text{nulo}\}}{N} \times 100
$$

onde $N$ é o número total de linhas e $j$ representa uma coluna.

In [5]:
# Calcula quantidade e percentual de faltantes por coluna
missing_count = df_raw.isna().sum()
missing_pct = (missing_count / len(df_raw) * 100).round(2)

resumo_missing = pd.DataFrame({
    'faltantes': missing_count,
    'percentual_faltantes': missing_pct
}).sort_values('percentual_faltantes', ascending=False)

print('Top 15 colunas com maior percentual de faltantes (base original):')
display(resumo_missing.head(15))

Top 15 colunas com maior percentual de faltantes (base original):


,faltantes,percentual_faltantes
TP_NOT,0,0.0
ID_AGRAVO,0,0.0
DT_NOTIFIC,0,0.0
SEM_NOT,0,0.0
NU_ANO,0,0.0
SG_UF_NOT,0,0.0
ID_MUNICIP,0,0.0
ID_REGIONA,0,0.0
ID_UNIDADE,0,0.0
DT_SIN_PRI,0,0.0


### Não há dados faltantes porque há campos com valores em branco ou ignorados, que devem ser considerados faltantes posteriormente.

## 2. Descarte Imediato de Colunas Irrelevantes

Removemos variáveis classificadas como:

- Constantes (sem poder discriminativo).
- Campos operacionais de fluxo/ (não há relevância para a análise).
- Campos de texto livre com alta ausência e baixa padronização.

In [6]:
# Lista consolidada a partir do descarte_imediato.txt
colunas_descarte_imediato = [
    'TP_NOT',      # constante
    'ID_AGRAVO',   # constante para B659 no dataset
    'DT_DIGITA',   # campo operacional de digitação
    'DT_TRANSUS',  # fluxo/sistema com ausência extrema
    'DT_TRANSDM',  # fluxo/sistema com ausência extrema
    'DT_TRANSSM',  # fluxo/sistema com alta ausência
    'DT_TRANSRS',  # fluxo/sistema com ausência extrema
    'DT_TRANSSE',  # fluxo/sistema com ausência extrema
    # texto livre com alta ausência
    'NOPROPIN',
    'NOCOLINF',
    'DS_FORMA',
    'OUTRO_EX'
]

colunas_existentes = [c for c in colunas_descarte_imediato if c in df_raw.columns]
colunas_nao_encontradas = [c for c in colunas_descarte_imediato if c not in df_raw.columns]

# Cria a base de trabalho removendo apenas colunas válidas
df = df_raw.drop(columns=colunas_existentes).copy()

print(f'Colunas removidas ({len(colunas_existentes)}): {colunas_existentes}')
print(f'Colunas do descarte que não estavam no CSV: {colunas_nao_encontradas}')
print(f'Nova dimensão da base: {df.shape[0]} linhas x {df.shape[1]} colunas')

Colunas removidas (12): ['TP_NOT', 'ID_AGRAVO', 'DT_DIGITA', 'DT_TRANSUS', 'DT_TRANSDM', 'DT_TRANSSM', 'DT_TRANSRS', 'DT_TRANSSE', 'NOPROPIN', 'NOCOLINF', 'DS_FORMA', 'OUTRO_EX']
Colunas do descarte que não estavam no CSV: []
Nova dimensão da base: 64345 linhas x 41 colunas


## 3. Padronização de Faltantes e Limpeza de Strings

Antes de qualquer imputação, padronizamos valores ausentes para um único padrão (`NaN`).

Isso evita que espaços, strings vazias e variações textuais sejam interpretadas como categorias válidas.

In [7]:
# 1) Converte células com apenas espaços para NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# 2) Em colunas textuais, remove espaços nas extremidades e reforça vazio -> NaN
for col in df.select_dtypes(include=['object']).columns:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .replace({'': pd.NA})
    )

print('Padronização de faltantes em campos textuais concluída.')

Padronização de faltantes em campos textuais concluída.


## 4. Conversão de Datas

A base contém datas em formatos mistos (por exemplo, `AAAA-MM-DD` e `AAAAMMDD`).

Para resolver isso, aplicamos uma função robusta de parsing em duas tentativas:
1. Primeiro, formato ISO (`%Y-%m-%d`).
2. Depois, formato numérico (`%Y%m%d`) para os casos que falharem.

In [8]:
def converter_data_robusta(serie: pd.Series) -> pd.Series:
    # Converte para string para tratar entradas heterogêneas e remove ruído textual
    s = (
        serie.astype('string')
        .str.strip()
        .replace({'': pd.NA, 'nan': pd.NA, 'None': pd.NA, 'NaT': pd.NA})
    )

    # Tentativa 1: datas no padrão ISO (YYYY-MM-DD)
    data_iso = pd.to_datetime(s, format='%Y-%m-%d', errors='coerce')

    # Tentativa 2: remove sufixo decimal típico de float (ex.: 20210102.0 -> 20210102)
    s_sem_decimal = s.str.replace(r'\.0+$', '', regex=True)
    data_num_str = pd.to_datetime(s_sem_decimal, format='%Y%m%d', errors='coerce')

    # Tentativa 3: rota numérica (inclui floats e notação científica)
    s_num = pd.to_numeric(
        s_sem_decimal.str.replace(',', '.', regex=False),
        errors='coerce'
    )
    s_num_int = pd.Series(pd.array(np.floor(s_num), dtype='Int64'), index=s.index).astype('string')
    data_num_float = pd.to_datetime(s_num_int, format='%Y%m%d', errors='coerce')

    # Combina estratégias de parsing
    return data_iso.fillna(data_num_str).fillna(data_num_float)

colunas_data = [
    'DT_NOTIFIC', 'DT_SIN_PRI', 'DT_INVEST', 'DT_COPRO',
    'DT_ENCERRA', 'DT_OBITO', 'DTTRAT', 'DT_RESU3'
]

for col in colunas_data:
    if col in df.columns:
        # Fallback: se a coluna já estiver parcialmente perdida (NaT), recupera valor bruto do df_raw
        serie_base = df[col].where(df[col].notna(), df_raw[col] if col in df_raw.columns else df[col])
        df[col] = converter_data_robusta(serie_base)

print('Conversão de datas concluída.')
display(df[colunas_data].head())

Conversão de datas concluída.


,DT_NOTIFIC,DT_SIN_PRI,DT_INVEST,DT_COPRO,DT_ENCERRA,DT_OBITO,DTTRAT,DT_RESU3
0,2007-10-30,2007-07-11,2007-11-12,2007-11-07,2007-12-26,NaT,2007-11-22,NaT
6,2007-10-08,2007-08-08,2007-10-08,2007-09-24,2007-10-09,NaT,NaT,NaT
7,2007-10-18,2007-08-15,2007-10-18,2007-10-08,2007-10-18,NaT,2007-10-18,NaT
8,2007-10-09,2007-08-10,2007-10-09,2007-08-29,2007-10-09,NaT,2007-10-09,NaT
9,2007-10-02,2007-09-27,2007-10-02,2007-09-28,2007-10-02,NaT,2007-10-02,NaT


In [9]:
# Validação da qualidade de conversão das colunas de data
resumo_conversao = []
amostras_falha = {}

for col in colunas_data:
    if col not in df.columns or col not in df_raw.columns:
        continue

    # Considera como "valor original válido" tudo que não é vazio/nulo textual
    original = df_raw[col].astype('string').str.strip()
    mascara_original_valido = (
        original.notna()
        & (original != '')
        & (~original.str.lower().isin(['nan', 'none']))
    )

    total_original_valido = int(mascara_original_valido.sum())
    total_convertido = int((df[col].notna() & mascara_original_valido).sum())
    total_nao_convertido = int(total_original_valido - total_convertido)

    taxa_conversao = (
        (total_convertido / total_original_valido) * 100
        if total_original_valido > 0
        else np.nan
    )

    resumo_conversao.append({
        'coluna_data': col,
        'valores_originais_validos': total_original_valido,
        'convertidos_com_sucesso': total_convertido,
        'nao_convertidos': total_nao_convertido,
        'taxa_conversao_%': round(taxa_conversao, 2) if pd.notna(taxa_conversao) else np.nan
    })

    if total_nao_convertido > 0:
        valores_problematicos = (
            original[mascara_original_valido & df[col].isna()]
            .drop_duplicates()
            .head(10)
            .tolist()
        )
        amostras_falha[col] = valores_problematicos

resumo_conversao_df = pd.DataFrame(resumo_conversao).sort_values('taxa_conversao_%')
print('Resumo da auditoria de conversão de datas:')
display(resumo_conversao_df)

if amostras_falha:
    print('\nAmostras de valores que não foram convertidos (máximo 10 por coluna):')
    for coluna, amostras in amostras_falha.items():
        print(f'- {coluna}: {amostras}')
else:
    print('\nTodas as colunas de data tiveram 100% de conversão para os valores originais válidos.')

Resumo da auditoria de conversão de datas:


,coluna_data,valores_originais_validos,convertidos_com_sucesso,nao_convertidos,taxa_conversao_%
0,DT_NOTIFIC,64345,64345,0,100.0
1,DT_SIN_PRI,64345,64345,0,100.0
2,DT_INVEST,64345,64345,0,100.0
3,DT_COPRO,64345,64345,0,100.0
4,DT_ENCERRA,63879,63879,0,100.0
5,DT_OBITO,257,257,0,100.0
6,DTTRAT,62367,62367,0,100.0
7,DT_RESU3,6582,6582,0,100.0



Todas as colunas de data tiveram 100% de conversão para os valores originais válidos.


### Agora temos atributos de data padronizados como objetos datetime, prontos para extração de features temporais.

## 5. Engenharia de Atributos Temporais

Criamos variáveis derivadas que ajudam a capturar dinâmica do caso clínico e do fluxo epidemiológico:

1. `delay_notificacao_dias`: diferença entre notificação e início de sintomas.
$$
\text{delay} = DT\_NOTIFIC - DT\_SIN\_PRI
$$

2. `tempo_encerramento_dias`: tempo total para encerramento do caso.
$$
\text{encerramento} = DT\_ENCERRA - DT\_NOTIFIC
$$

3. `idade_aprox_no_evento`: aproximação da idade no momento da notificação.
$$
\text{idade\_aprox} = DT\_NOTIFIC - ANO\_NASC
$$

In [10]:
# Cria cópias derivadas somente quando as colunas-base existem
if {'DT_NOTIFIC', 'DT_SIN_PRI'}.issubset(df.columns):
    df['delay_notificacao_dias'] = (df['DT_NOTIFIC'] - df['DT_SIN_PRI']).dt.days

if {'DT_ENCERRA', 'DT_NOTIFIC'}.issubset(df.columns):
    df['tempo_encerramento_dias'] = (df['DT_ENCERRA'] - df['DT_NOTIFIC']).dt.days

if {'DT_NOTIFIC', 'ANO_NASC'}.issubset(df.columns):
    # Converte ANO_NASC para número antes da operação
    df['ANO_NASC'] = pd.to_numeric(df['ANO_NASC'], errors='coerce')
    df['idade_aprox_no_evento'] = df['DT_NOTIFIC'].dt.year - df['ANO_NASC']

print('Features temporais criadas.')
colunas_criadas = [c for c in ['delay_notificacao_dias', 'tempo_encerramento_dias', 'idade_aprox_no_evento'] if c in df.columns]
display(df[colunas_criadas].describe().T)

Features temporais criadas.


,count,mean,std,min,25%,50%,75%,max
delay_notificacao_dias,64345.0,97.386961,671.18675,0.0,1.0,23.0,64.0,29160.0
tempo_encerramento_dias,63879.0,57.097246,307.679087,0.0,0.0,15.0,61.0,73048.0
idade_aprox_no_evento,62573.0,33.268454,17.628635,0.0,19.0,31.0,45.0,121.0


In [11]:
# Amostrar 10 linhas com maiores delays para inspeção
if 'delay_notificacao_dias' in df.columns:
    print('10 casos com maiores delays entre início dos sintomas e notificação:')
    display(df.sort_values('delay_notificacao_dias', ascending=False).head(10)[['DT_SIN_PRI', 'DT_NOTIFIC', 'delay_notificacao_dias', 'ANO_NASC']])


10 casos com maiores delays entre início dos sintomas e notificação:


,DT_SIN_PRI,DT_NOTIFIC,delay_notificacao_dias,ANO_NASC
108748,1932-04-17,2012-02-17,29160,1932
66216,1935-02-11,2010-01-27,27379,1935
163781,1953-09-22,2024-04-09,25767,1953
104309,1943-07-03,2011-09-14,24910,1943
99552,1945-04-15,2011-05-24,24145,1945
152972,1956-02-03,2020-02-13,23386,1956
107245,1948-07-28,2011-12-13,23148,<NA>
94853,1947-11-11,2011-02-14,23106,1947
68416,1947-08-28,2010-03-08,22838,1947
128923,1954-11-05,2014-12-29,21969,1954


### O ponto de atenção é que existem casos com `delay_notificacao_dias` muito alto, o que pode significar um erro de registro onde o ano de nascimento foi registrado como data de primeiros sintomas.

## 6. Regras de Consistência de Domínio

Com base no contexto epidemiológico, aplicamos regras simples para reduzir inconsistências:
- `NU_IDADE_N` deve estar em $[0, 130]$.
- `ANO_NASC` deve estar entre 1900 e ano atual.
- `AN_QUANT` (conforme dicionário utilizado) deve ser 0 ou 1.

Valores fora dessas faixas são convertidos para `NaN` e tratados na etapa de imputação.

In [12]:
# 1. Garantir que as colunas sejam numéricas (float)
for col in ['NU_IDADE_N', 'ANO_NASC', 'AN_QUANT']:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Lógica da Idade SINAN (4024 -> 24 anos)
if 'NU_IDADE_N' in df.columns:
    # Calculamos unidade e valor garantindo que sejam tipos numéricos básicos
    unidade = (df['NU_IDADE_N'] // 1000).fillna(0).astype(int)
    valor = (df['NU_IDADE_N'] % 1000).fillna(0)

    # Criamos as condições usando .values para evitar o erro de boolean do Pandas
    condlist = [
        (unidade == 4).values, # Anos
        (unidade == 3).values, # Meses
        (unidade < 3).values   # Dias/Horas
    ]

    choicelist = [
        valor,        # Se anos, mantém o valor
        valor / 12,   # Se meses, divide por 12
        0             # Se dias/horas, considera 0 anos para o modelo
    ]

    # Aplicamos o select
    df['IDADE_PROCESSADA'] = np.select(condlist, choicelist, default=np.nan)

    # Validação final (limites humanos)
    mask_invalida = (df['IDADE_PROCESSADA'] < 0) | (df['IDADE_PROCESSADA'] > 130)
    qtd_idade_invalida = mask_invalida.sum()
    df.loc[mask_invalida, 'IDADE_PROCESSADA'] = np.nan

    # Atualiza a coluna principal
    df['NU_IDADE_N'] = df['IDADE_PROCESSADA']
else:
    qtd_idade_invalida = 0

# --- As demais correções permanecem ---
ano_atual = pd.Timestamp.today().year
if 'ANO_NASC' in df.columns:
    mask_ano = (df['ANO_NASC'] < 1900) | (df['ANO_NASC'] > ano_atual)
    qtd_ano_invalido = mask_ano.sum()
    df.loc[mask_ano, 'ANO_NASC'] = np.nan
else:
    qtd_ano_invalido = 0

# Nota sobre AN_QUANT:
# Na esquistossomose, AN_QUANT é a carga parasitária (ovos por grama).
# Se você forçar apenas 0 e 1, perderá a informação de intensidade da doença!
if 'AN_QUANT' in df.columns:
    # Removi a restrição de [0, 1] para manter a contagem de ovos real
    mask_negativo = (df['AN_QUANT'] < 0)
    qtd_anquant_invalido = mask_negativo.sum()
    df.loc[mask_negativo, 'AN_QUANT'] = np.nan
else:
    qtd_anquant_invalido = 0

print(f'Idades SINAN (ex: 4024 -> 24) processadas: {len(df) - df["NU_IDADE_N"].isna().sum()}')
print(f'Inconsistências (0-130 anos) limpas: {qtd_idade_invalida}')
print(f'Anos de nascimento inválidos corrigidos: {qtd_ano_invalido}')

Idades SINAN (ex: 4024 -> 24) processadas: 64345
Inconsistências (0-130 anos) limpas: 0
Anos de nascimento inválidos corrigidos: 3


## 7. Tratamento de Códigos Categóricos de Ignorado

No SINAN, alguns campos usam códigos de ignorado (por exemplo: 9, 99, 999, `I`).

Para não misturar ausência de informação com categoria real, mapeamos esses códigos para `NaN` em colunas selecionadas.

In [13]:
colunas_ignorado = [
    'CS_SEXO', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'OUTROS',
    'TRATAM', 'TRATANAO', 'STCURA1', 'STCURA2', 'STCURA3',
    'FORMA', 'TPAUTOCTO', 'DOENCA_TRA', 'EVOLUCAO', 'AN_QUALI'
]

mapa_ignorado = {
    'I': pd.NA,
    '9': pd.NA,
    '99': pd.NA,
    '999': pd.NA,
    '9999': pd.NA,
    'Ignorado': pd.NA,
    'IGNORADO': pd.NA
}

for col in [c for c in colunas_ignorado if c in df.columns]:
    df[col] = (
        df[col]
        .astype('string')
        .str.strip()
        .replace(mapa_ignorado)
    )

print('Códigos de ignorado padronizados para NaN nas colunas categóricas selecionadas.')

Códigos de ignorado padronizados para NaN nas colunas categóricas selecionadas.


## 8. Consolidação de Base para Modelagem

Nesta etapa criamos uma base específica para modelagem (`df_model`)

In [14]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 64345 entries, 0 to 166857
Data columns (total 45 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   DT_NOTIFIC               64345 non-null  datetime64[ns]
 1   SEM_NOT                  64345 non-null  string        
 2   NU_ANO                   64345 non-null  string        
 3   SG_UF_NOT                64345 non-null  string        
 4   ID_MUNICIP               64345 non-null  string        
 5   ID_REGIONA               63675 non-null  string        
 6   ID_UNIDADE               64344 non-null  string        
 7   DT_SIN_PRI               64345 non-null  datetime64[ns]
 8   SEM_PRI                  64345 non-null  string        
 9   ANO_NASC                 62570 non-null  Int64         
 10  NU_IDADE_N               64345 non-null  float64       
 11  CS_SEXO                  64340 non-null  string        
 12  CS_GESTANT               61684 non-n

In [15]:
# 1. Drop de colunas inúteis ou muito vazias
cols_to_drop = ['TRATANAO', 'DT_OBITO', 'DT_RESU3',

    # 2. IDs e Administrativas (Alta cardinalidade - "ruído")
    'ID_REGIONA', 'ID_MN_RESI', 'ID_RG_RESI', 'SEM_NOT', 'SEM_PRI', 'NU_ANO',
    'ID_OCUPA_N', 'COMUNINF', 'OUTROS','SG_UF_NOT', 'ID_PAIS',

    # 3. Datas Puras (Já temos as colunas de "delay" e "tempo")
    'DT_NOTIFIC', 'DT_SIN_PRI', 'DT_INVEST', 'DT_ENCERRA',
    'DTTRAT', 'DT_COPRO',

    # 4. Redundâncias de Idade (Manteremos apenas a IDADE_PROCESSADA)
    'NU_IDADE_N', 'ANO_NASC', 'idade_aprox_no_evento',

    # 5. Remoção de STCURA e AN_QUANT para evitar data leakage já que o target é evolução do caso (cura, óbito, etc) e se há 0 ovos, então o caso já está curado
    'STCURA1', 'STCURA2', 'STCURA3', 'AN_QUANT',

    # 6. TPAUTOCTO (o caso é autoctone do municipio de residencia) não é relevante para prever evolução
    'TPAUTOCTO',

    # 7. COPAISINF (indica o pais de fonte da infecção) mas a maioria dos casos é Brasil
    'COPAISINF',
] 
df_model = df.drop(columns=cols_to_drop)

# 2. Imputação Categórica (9 = Ignorado)
cat_cols = ['CS_RACA', 'CS_ESCOL_N', 'DOENCA_TRA']
# Em CS_RACA, 9 = Ignorado

# Em CS_ESCOL_N, 0 = Analfabeto, 1 = Fundamental incompleto, 2 = Fundamental completo, 3 = Fundamental II incompleto, 4 = Fundamental II Completo, 5 = Ensino Médio Incompleto,
#  6 = Ensino Médio Completo, 7 = Ensino Superior Incompleto, 8 = Ensino Superior Completo, 9 = Ignorado, 10 = Não se Aplica

# Em FORMA, 1 = Intestinal, 2 = Hepato intestinal, 3 = Hepato esplênica, 4 = Aguda, 5 = Outra

# Em TPAUTOCTO (o caso é autoctone), 1 = sim, 2 = não, 3 = indeterminado

# Em DOENCA_TRA, 1 = Caso Relacionado ao Trabalho, 2 = Não Relacionado ao Trabalho, 9 = Ignorado

# Em STCURA 1, 0 = 0 ovos, 1 = 1 ou mais ovos, 2 = Não Realizado
# Em STCURA 2, 0 = 0 ovos, 1 = 1 ou mais ovos, 2 = Não Realizado
# Em STCURA 3, 0 = 0 ovos, 1 = 1 ou mais ovos, 2 = Não Realizado


for col in cat_cols:
    df_model[col] = df_model[col].fillna('9')

# 3. Imputação Numérica (Mediana)
num_cols = ['tempo_encerramento_dias', 'IDADE_PROCESSADA']
for col in num_cols:
    df_model[col] = df_model[col].fillna(df_model[col].median())

In [16]:
print(f"Limpeza concluída!")
print(f"Colunas restantes: {df_model.shape[1]}")
print("-" * 30)
print(df_model.columns.tolist())

Limpeza concluída!
Colunas restantes: 16
------------------------------
['ID_MUNICIP', 'ID_UNIDADE', 'CS_SEXO', 'CS_GESTANT', 'CS_RACA', 'CS_ESCOL_N', 'SG_UF', 'TRATAM', 'FORMA', 'COUFINF', 'DOENCA_TRA', 'EVOLUCAO', 'AN_QUALI', 'delay_notificacao_dias', 'tempo_encerramento_dias', 'IDADE_PROCESSADA']


In [17]:
df_model.info()

# salvar df_model em .csv em ../data/processed/esqu_model_ready.csv
df_model.to_csv('../data/esqu_model_ready.csv', index=False)

<class 'pandas.core.frame.DataFrame'>
Index: 64345 entries, 0 to 166857
Data columns (total 16 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID_MUNICIP               64345 non-null  string 
 1   ID_UNIDADE               64344 non-null  string 
 2   CS_SEXO                  64340 non-null  string 
 3   CS_GESTANT               61684 non-null  string 
 4   CS_RACA                  64345 non-null  string 
 5   CS_ESCOL_N               64345 non-null  string 
 6   SG_UF                    64344 non-null  string 
 7   TRATAM                   63697 non-null  string 
 8   FORMA                    51809 non-null  string 
 9   COUFINF                  55802 non-null  string 
 10  DOENCA_TRA               64345 non-null  string 
 11  EVOLUCAO                 64345 non-null  string 
 12  AN_QUALI                 60097 non-null  string 
 13  delay_notificacao_dias   64345 non-null  int64  
 14  tempo_encerramento_dias  6

## 9. Codificação e Padronização para Modelagem

Nesta seção transformamos a base `df_model` em matrizes numéricas prontas para algoritmos de Machine Learning, mantendo o atributo-alvo `EVOLUCAO` separado dos preditores.

### 9.1 Objetivo da preparação
- Garantir entrada consistente para os modelos.
- Reduzir risco de vazamento de dados (data leakage).
- Controlar a dimensionalidade das variáveis categóricas.

### 9.2 Estratégia adotada

A preparação foi desenhada em quatro blocos principais:

1. Separação entre preditores (`X`) e alvo (`y`).
2. Divisão treino/teste com estratificação do alvo.
3. Codificação híbrida para variáveis categóricas:
   - baixa cardinalidade: One-Hot Encoding;
   - alta cardinalidade: Frequency Encoding.
4. Padronização de atributos numéricos para estabilizar escalas.

### 9.3 Frequência como representação de categoria

Para categorias com muitas classes, substituímos cada categoria por sua frequência relativa no treino:
$$
\operatorname{Freq}(c) = \frac{n_c}{N},
$$
onde $n_c$ é o número de ocorrências da categoria $c$ no treino e $N$ é o tamanho do treino.

Essa abordagem reduz dimensionalidade e preserva informação de prevalência sem criar milhares de colunas binárias.

#### 9.3.1 Separação, estratificação e higiene de tipos

Na próxima célula, o código:

1. Separa `X` e `y` (com `EVOLUCAO` como alvo).
2. Cria treino/teste estratificado para preservar proporções de classe.
3. Higieniza tipos e faltantes para compatibilidade com scikit-learn.
4. Detecta colunas de alta cardinalidade para aplicar Frequency Encoding.

In [18]:
# Imports específicos da etapa de preparação para modelagem
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# 1) Define alvo e garante consistência de tipo
target_col = 'EVOLUCAO'
if target_col not in df_model.columns:
    raise KeyError(f"A coluna alvo '{target_col}' não está em df_model.")

base_modelagem = df_model.copy()
base_modelagem[target_col] = pd.to_numeric(base_modelagem[target_col], errors='coerce')
base_modelagem = base_modelagem.dropna(subset=[target_col]).copy()
base_modelagem[target_col] = base_modelagem[target_col].astype(int)

# 2) Separa preditores (X) e alvo (y)
X = base_modelagem.drop(columns=[target_col])
y = base_modelagem[target_col]

# 3) Split treino/teste estratificado para preservar proporção das classes
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
 )

# 4) Detecta tipos de coluna com base no treino
colunas_numericas = X_train.select_dtypes(include=['number']).columns.tolist()
colunas_categoricas = [col for col in X_train.columns if col not in colunas_numericas]

# 5) Sanitização robusta para evitar pd.NA dentro do scikit-learn
# Numéricas: converte para número e força ausentes para np.nan
for col in colunas_numericas:
    X_train[col] = pd.to_numeric(X_train[col], errors='coerce')
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# Categóricas: força dtype object e substitui qualquer ausente por np.nan
for col in colunas_categoricas:
    X_train[col] = X_train[col].astype('object').where(X_train[col].notna(), np.nan)
    X_test[col] = X_test[col].astype('object').where(X_test[col].notna(), np.nan)

# 6) Estratégia híbrida por cardinalidade
limiar_alta_cardinalidade = 30
cardinalidades = {col: int(X_train[col].nunique(dropna=True)) for col in colunas_categoricas}
colunas_alta_card = [
    col for col, n_cat in cardinalidades.items()
    if n_cat > limiar_alta_cardinalidade
]
colunas_baixa_card = [col for col in colunas_categoricas if col not in colunas_alta_card]

# 7) Frequency Encoding para colunas de alta cardinalidade (ajuste no treino)
mapas_frequencia = {}
X_train_modelagem = X_train.copy()
X_test_modelagem = X_test.copy()

for col in colunas_alta_card:
    treino_cat = X_train_modelagem[col].fillna('__MISSING__')
    teste_cat = X_test_modelagem[col].fillna('__MISSING__')

    freq_map = treino_cat.value_counts(normalize=True)
    mapas_frequencia[col] = freq_map

    X_train_modelagem[col] = treino_cat.map(freq_map).astype(float)
    X_test_modelagem[col] = teste_cat.map(freq_map).fillna(0.0).astype(float)

# 8) Listas finais para o ColumnTransformer
colunas_numericas_modelagem = sorted(set(colunas_numericas + colunas_alta_card))
colunas_categoricas_modelagem = colunas_baixa_card

print(f"Registros totais para modelagem: {len(base_modelagem):,}")
print(f"Treino: {X_train.shape[0]:,} linhas | Teste: {X_test.shape[0]:,} linhas")
print(f"Colunas numéricas originais: {len(colunas_numericas)}")
print(f"Colunas categóricas totais: {len(colunas_categoricas)}")
print(f"Alta cardinalidade (Frequency Encoding): {len(colunas_alta_card)}")
print(f"Baixa cardinalidade (One-Hot): {len(colunas_baixa_card)}")

if colunas_alta_card:
    resumo_card = pd.Series({c: cardinalidades[c] for c in colunas_alta_card}).sort_values(ascending=False)
    print('\nTop colunas de alta cardinalidade:')
    display(resumo_card.head(10))

Registros totais para modelagem: 64,345
Treino: 51,476 linhas | Teste: 12,869 linhas
Colunas numéricas originais: 3
Colunas categóricas totais: 12
Alta cardinalidade (Frequency Encoding): 2
Baixa cardinalidade (One-Hot): 10

Top colunas de alta cardinalidade:


ID_UNIDADE    5384
ID_MUNICIP    1473
dtype: int64

### 9.4 Pipeline de transformação (imputação, codificação e escala)

Após decidir quais colunas usam Frequency Encoding e quais usam One-Hot, montamos um `ColumnTransformer` em dois ramos:

- **Ramo numérico**
  - imputação por mediana;
  - padronização com `StandardScaler`.

- **Ramo categórico de baixa cardinalidade**
  - imputação pela moda;
  - codificação com `OneHotEncoder(handle_unknown='ignore')`.

In [19]:
# Configura codificador categórico com compatibilidade entre versões do scikit-learn
try:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown='ignore', sparse=False)

# Pipeline para colunas numéricas (inclui frequência de alta cardinalidade)
pipeline_numerico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para colunas categóricas de baixa cardinalidade
pipeline_categorico = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', encoder)
])

# Monta transformadores de forma dinâmica
transformers = [
    ('num', pipeline_numerico, colunas_numericas_modelagem),
]
if colunas_categoricas_modelagem:
    transformers.append(('cat', pipeline_categorico, colunas_categoricas_modelagem))

# Combina os fluxos em um único pré-processador
preprocessador = ColumnTransformer(
    transformers=transformers,
    remainder='drop'
 )

# Ajusta somente no treino e aplica em treino/teste
X_train_prepared = preprocessador.fit_transform(X_train_modelagem)
X_test_prepared = preprocessador.transform(X_test_modelagem)

# Organiza as saídas em DataFrames para inspeção e rastreabilidade
nomes_atributos = preprocessador.get_feature_names_out()
X_train_prepared = pd.DataFrame(X_train_prepared, columns=nomes_atributos, index=X_train.index)
X_test_prepared = pd.DataFrame(X_test_prepared, columns=nomes_atributos, index=X_test.index)

print(f"Formato final X_train_prepared: {X_train_prepared.shape}")
print(f"Formato final X_test_prepared: {X_test_prepared.shape}")

Formato final X_train_prepared: (51476, 102)
Formato final X_test_prepared: (12869, 102)


#### 9.4.1 Interpretação do formato final

> A dimensionalidade final (`X_train_prepared.shape[1]`) representa o total de atributos efetivamente entregues aos modelos.

A redução desse número, mantendo informação relevante, melhora:
- desempenho computacional;
- estabilidade no treino;
- legibilidade do pipeline.

### 9.5 Checagens finais e artefatos de modelagem

Antes de seguir para PCA e treinamento, validamos:

1. **Estratificação do alvo** no treino e no teste (proporções semelhantes).
2. **Estrutura da matriz final** após codificação e escala.
3. **Persistência dos artefatos** no dicionário `dados_modelagem`.

Esse dicionário concentra os objetos necessários para as próximas etapas:
- `X_train`, `X_test`, `y_train`, `y_test`;
- `preprocessador` já ajustado.

In [ ]:
# Função auxiliar para exibir proporções de classe
def resumo_classes(y_serie: pd.Series, nome: str) -> pd.DataFrame:
    return (
        y_serie.value_counts(normalize=True)
        .sort_index()
        .rename(nome)
        .mul(100)
        .round(2)
        .to_frame()
    )

print('Distribuição percentual das classes de EVOLUCAO:')
display(pd.concat([
    resumo_classes(y_train, 'treino_%'),
    resumo_classes(y_test, 'teste_%')
], axis=1))

#print('Amostra das colunas geradas após codificação:')
#display(pd.Series(X_train_prepared.columns).head(20))

# Dicionário consolidado para reutilizar nas etapas de treinamento e avaliação
dados_modelagem = {
    'X_train': X_train_prepared,
    'X_test': X_test_prepared,
    'y_train': y_train,
    'y_test': y_test,
    'preprocessador': preprocessador
}

print('Objeto dados_modelagem criado com sucesso.')
print(f"Total de atributos prontos para modelagem: {X_train_prepared.shape[1]}")

Distribuição percentual das classes de EVOLUCAO:


,treino_%,teste_%
EVOLUCAO,,
1,98.34,98.34
2,1.26,1.26
3,0.40,0.40


Amostra das colunas geradas após codificação:


0            num__IDADE_PROCESSADA
1                  num__ID_MUNICIP
2                  num__ID_UNIDADE
3      num__delay_notificacao_dias
4     num__tempo_encerramento_dias
5                   cat__CS_SEXO_F
6                   cat__CS_SEXO_M
7                cat__CS_GESTANT_1
8                cat__CS_GESTANT_2
9                cat__CS_GESTANT_3
10               cat__CS_GESTANT_4
11               cat__CS_GESTANT_5
12               cat__CS_GESTANT_6
13                  cat__CS_RACA_1
14                  cat__CS_RACA_2
15                  cat__CS_RACA_3
16                  cat__CS_RACA_4
17                  cat__CS_RACA_5
18                  cat__CS_RACA_9
19               cat__CS_ESCOL_N_0
dtype: object

Objeto dados_modelagem criado com sucesso.
Total de atributos prontos para modelagem: 102


In [21]:
X_train_prepared.head(10)

,num__IDADE_PROCESSADA,num__ID_MUNICIP,num__ID_UNIDADE,num__delay_notificacao_dias,num__tempo_encerramento_dias,cat__CS_SEXO_F,cat__CS_SEXO_M,cat__CS_GESTANT_1,cat__CS_GESTANT_2,cat__CS_GESTANT_3,cat__CS_GESTANT_4,cat__CS_GESTANT_5,cat__CS_GESTANT_6,cat__CS_RACA_1,cat__CS_RACA_2,cat__CS_RACA_3,cat__CS_RACA_4,cat__CS_RACA_5,cat__CS_RACA_9,cat__CS_ESCOL_N_0,cat__CS_ESCOL_N_00,cat__CS_ESCOL_N_01,cat__CS_ESCOL_N_02,cat__CS_ESCOL_N_03,cat__CS_ESCOL_N_04,cat__CS_ESCOL_N_05,cat__CS_ESCOL_N_06,cat__CS_ESCOL_N_07,cat__CS_ESCOL_N_08,cat__CS_ESCOL_N_09,cat__CS_ESCOL_N_1,cat__CS_ESCOL_N_10,cat__CS_ESCOL_N_3,cat__CS_ESCOL_N_5,cat__CS_ESCOL_N_8,cat__CS_ESCOL_N_9,cat__SG_UF_11,cat__SG_UF_13,cat__SG_UF_14,cat__SG_UF_15,cat__SG_UF_17,cat__SG_UF_21,cat__SG_UF_22,cat__SG_UF_23,cat__SG_UF_24,cat__SG_UF_25,cat__SG_UF_26,cat__SG_UF_27,cat__SG_UF_28,cat__SG_UF_29,cat__SG_UF_31,cat__SG_UF_32,cat__SG_UF_33,cat__SG_UF_35,cat__SG_UF_41,cat__SG_UF_42,cat__SG_UF_43,cat__SG_UF_50,cat__SG_UF_51,cat__SG_UF_52,cat__SG_UF_53,cat__TRATAM_1,cat__TRATAM_2,cat__TRATAM_3,cat__TRATAM_4,cat__FORMA_1,cat__FORMA_2,cat__FORMA_3,cat__FORMA_4,cat__FORMA_5,cat__COUFINF_11,cat__COUFINF_12,cat__COUFINF_13,cat__COUFINF_14,cat__COUFINF_15,cat__COUFINF_17,cat__COUFINF_21,cat__COUFINF_22,cat__COUFINF_23,cat__COUFINF_24,cat__COUFINF_25,cat__COUFINF_26,cat__COUFINF_27,cat__COUFINF_28,cat__COUFINF_29,cat__COUFINF_31,cat__COUFINF_32,cat__COUFINF_33,cat__COUFINF_35,cat__COUFINF_41,cat__COUFINF_42,cat__COUFINF_43,cat__COUFINF_50,cat__COUFINF_51,cat__COUFINF_52,cat__COUFINF_53,cat__DOENCA_TRA_1,cat__DOENCA_TRA_2,cat__DOENCA_TRA_9,cat__AN_QUALI_1,cat__AN_QUALI_2,cat__AN_QUALI_3
64417,-0.648531,-0.851940,-0.647827,-0.124854,-0.145405,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
13161,-0.024686,-0.851940,-0.664561,-0.144107,0.477524,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
96713,0.429020,-0.448317,-0.508377,-0.075980,-0.169023,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
99747,-0.364965,0.421905,-0.664561,-0.087828,-0.107026,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
71585,-0.081399,0.029733,0.651847,-0.138183,0.604472,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0
98426,-0.081399,-0.860527,-0.681295,-0.015258,0.512951,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0

## 10. Redução de Dimensionalidade com PCA

Nesta seção aplicamos PCA aos dados já processados na seção 9 para compactar informação sem perder grande parte da variabilidade.

### 10.1 Intuição teórica
O PCA projeta os dados em novas direções ortogonais (componentes principais) que maximizam variância.

### 10.2 Critério de retenção
Usamos retenção de variância acumulada mínima de 95%:
$$
\sum_{k=1}^{m} \text{Var}(PC_k) \ge 0.95
$$
onde $m$ é o número de componentes retidos.

### 10.3 Boas práticas aplicadas
- Ajustar PCA somente no treino.
- Aplicar a mesma transformação no teste.
- Registrar componentes e variância explicada para rastreabilidade.

In [22]:
# Validação mínima de pré-requisito da seção 9
if 'X_train_prepared' not in globals() or 'X_test_prepared' not in globals():
    raise RuntimeError("Execute a seção 9 antes do PCA: X_train_prepared e X_test_prepared não encontrados.")

# 1) Define o PCA para preservar 95% da variância
variancia_alvo = 0.95
pca_model = PCA(n_components=variancia_alvo, svd_solver='full')

# 2) Ajusta no treino e transforma treino/teste
X_train_pca_array = pca_model.fit_transform(X_train_prepared)
X_test_pca_array = pca_model.transform(X_test_prepared)

# 3) Organiza em DataFrames para manter rastreabilidade
n_componentes = X_train_pca_array.shape[1]
colunas_pca = [f'PC{i+1}' for i in range(n_componentes)]

X_train_pca = pd.DataFrame(X_train_pca_array, columns=colunas_pca, index=X_train_prepared.index)
X_test_pca = pd.DataFrame(X_test_pca_array, columns=colunas_pca, index=X_test_prepared.index)

# 4) Resumo de variância explicada
var_exp = pca_model.explained_variance_ratio_
var_acum = np.cumsum(var_exp)

resumo_pca = pd.DataFrame({
    'componente': colunas_pca,
    'variancia_explicada': var_exp,
    'variancia_acumulada': var_acum
})

print('Resumo PCA:')
print(f'- Atributos antes do PCA: {X_train_prepared.shape[1]}')
print(f'- Componentes após PCA: {n_componentes}')
print(f'- Variância total explicada: {var_exp.sum():.4f}')

print('\nPrimeiros componentes:')
display(resumo_pca.head(10))

# 5) Salva artefatos no dicionário de modelagem
if 'dados_modelagem' in globals():
    dados_modelagem['X_train_pca'] = X_train_pca
    dados_modelagem['X_test_pca'] = X_test_pca
    dados_modelagem['pca_model'] = pca_model
    print('\nPCA adicionado em dados_modelagem: X_train_pca, X_test_pca e pca_model.')
else:
    print('\nAviso: dados_modelagem não existe; PCA gerado apenas nas variáveis X_train_pca/X_test_pca.')

Resumo PCA:
- Atributos antes do PCA: 102
- Componentes após PCA: 26
- Variância total explicada: 0.9549

Primeiros componentes:


,componente,variancia_explicada,variancia_acumulada
0,PC1,0.159991,0.159991
1,PC2,0.109890,0.269881
2,PC3,0.103457,0.373337
3,PC4,0.100069,0.473407
4,PC5,0.078145,0.551551
5,PC6,0.061920,0.613471
6,PC7,0.047233,0.660705
7,PC8,0.039463,0.700168
8,PC9,0.036284,0.736452
9,PC10,0.028518,0.764970



PCA adicionado em dados_modelagem: X_train_pca, X_test_pca e pca_model.


#### 10.4 Como interpretar a saída do PCA

Após executar a célula de PCA, observe:

1. **Atributos antes vs componentes depois**: mede compressão dimensional.
2. **Variância total explicada**: deve ficar próxima ou acima de 0.95 (critério adotado).
3. **Tabela dos primeiros componentes**: mostra contribuição incremental e acumulada.

Se a perda de desempenho preditivo for pequena, a versão com PCA tende a ser vantajosa em custo computacional.

## 12. Conclusão e Próximos Passos

### 12.1 O que foi consolidado
- Pré-processamento completo com limpeza, regras de domínio e imputação.
- Codificação híbrida (One-Hot + Frequency Encoding) para equilibrar informação e dimensionalidade.
- Padronização numérica e criação de base pronta para modelagem supervisionada.
- Versão alternativa com PCA para comparação de desempenho.

### 12.2 Principais artefatos gerados
- `dados_modelagem['X_train']` e `dados_modelagem['X_test']`: base sem PCA.
- `dados_modelagem['X_train_pca']` e `dados_modelagem['X_test_pca']`: base com PCA.
- `dados_modelagem['preprocessador']` e `dados_modelagem['pca_model']`: transformadores ajustados.

### 12.3 Próximos passos sugeridos
1. Treinar modelos baseline sem PCA e com PCA.
2. Comparar métricas por validação estratificada (ex.: F1 macro, recall por classe).
3. Ajustar hiperparâmetros e estratégia para desbalanceamento de classes.
4. Escolher o pipeline final com melhor equilíbrio entre desempenho e interpretabilidade.